# 01 Data Quality Assessment (NovaCred Credit Applications)

This notebook performs a comprehensive data quality assessment and remediation on the NovaCred credit applications dataset.

## Data quality dimensions assessed
- **Completeness**: missing values in critical fields
- **Uniqueness**: duplicate primary keys and duplicate records
- **Consistency**: standardised category values (for example gender), consistent types
- **Validity**: values respect logical rules and ranges
- **Accuracy**: format checks for fields like email, SSN, IP address, ZIP code

## Deliverables
- Quantified issue metrics (counts and percentages)
- Examples of affected records (IDs)
- Remediated dataset with clean columns and flags

In [0]:
from pyspark.sql import functions as F

input_path = "/Volumes/workspace/default/raw_data/raw_credit_applications.json"

# Read JSON safely (multiline is correct for your file)
df_raw = (
    spark.read
        .option("multiline", "true")
        .option("mode", "PERMISSIVE")
        .json(input_path)
)

# Print schema
df_raw.printSchema()

# If corrupt column exists, filter it out, otherwise just use df_raw
if "_corrupt_record" in df_raw.columns:
    corrupt_count = df_raw.filter(F.col("_corrupt_record").isNotNull()).count()
    total_rows = df_raw.count()
    print("Total rows read:", total_rows)
    print("Corrupt rows:", corrupt_count)

    df = df_raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
else:
    df = df_raw
    print("Total rows read:", df.count())

display(df.limit(5))

root
 |-- _id: string (nullable = true)
 |-- applicant_info: struct (nullable = true)
 |    |-- date_of_birth: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- full_name: string (nullable = true)
 |    |-- gender: string (nullable = true)
 |    |-- ip_address: string (nullable = true)
 |    |-- ssn: string (nullable = true)
 |    |-- zip_code: string (nullable = true)
 |-- decision: struct (nullable = true)
 |    |-- approved_amount: long (nullable = true)
 |    |-- interest_rate: double (nullable = true)
 |    |-- loan_approved: boolean (nullable = true)
 |    |-- rejection_reason: string (nullable = true)
 |-- financials: struct (nullable = true)
 |    |-- annual_income: string (nullable = true)
 |    |-- annual_salary: long (nullable = true)
 |    |-- credit_history_months: long (nullable = true)
 |    |-- debt_to_income: double (nullable = true)
 |    |-- savings_balance: long (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- notes: stri

_id,applicant_info,decision,financials,loan_purpose,notes,processing_timestamp,spending_behavior
app_200,"List(2001-03-09, jerry.smith17@hotmail.com, Jerry Smith, Male, 192.168.48.155, 596-64-4340, 10036)","List(null, null, false, algorithm_risk_score)","List(73000, null, 23, 0.2, 31212)",null,null,2024-01-15T00:00:00Z,"List(List(480, Shopping), List(790, Rent), List(247, Alcohol))"
app_037,"List(1992-03-31, brandon.walker2@yahoo.com, Brandon Walker, M, 10.1.102.112, 425-69-4784, 10032)","List(null, null, false, algorithm_risk_score)","List(78000, null, 51, 0.18, 17915)",null,null,null,"List(List(608, Rent), List(96, Dining), List(243, Healthcare))"
app_215,"List(1989-10-24, scott.moore94@mail.com, Scott Moore, Male, 10.240.193.250, 370-78-5178, 10075)","List(59000, 3.7, true, null)","List(61000, null, 41, 0.21, 37909)",vacation,null,null,"List(List(109, Rent))"
app_024,"List(1983-04-25, thomas.lee6@protonmail.com, Thomas Lee, Male, 192.168.175.67, 194-35-1833, 10077)","List(34000, 4.3, true, null)","List(103000, null, 70, 0.35, 0)",null,null,null,"List(List(575, Fitness))"
app_184,"List(1999-05-21, brian.rodriguez86@aol.com, Brian Rodriguez, M, 172.29.125.105, 480-41-2475, 10080)","List(null, null, false, algorithm_risk_score)","List(57000, null, 14, 0.23, 31763)",null,null,2024-01-15T00:00:00Z,"List(List(463, Entertainment))"



##STEP 1A — COMPLETENESS CHECK
 Evaluate missing values in critical fields


In [0]:

from pyspark.sql import functions as F

total = df.count()

df.select(
    F.count("*").alias("total_rows"),

    # Nested financial field
    F.sum(F.col("financials.annual_income").isNull().cast("int"))
        .alias("null_annual_income"),

    F.sum(F.col("applicant_info.date_of_birth").isNull().cast("int"))
        .alias("null_date_of_birth"),

    F.sum(F.col("applicant_info.email").isNull().cast("int"))
        .alias("null_email"),

    F.sum(F.col("decision.loan_approved").isNull().cast("int"))
        .alias("null_loan_approved")

).display()

total_rows,null_annual_income,null_date_of_birth,null_email,null_loan_approved
502,5,1,0,0


In [0]:
# Check nulls for loan_approved only
df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("decision.loan_approved").isNull().cast("int")).alias("null_loan_approved")
).display()

total_rows,null_loan_approved
502,0


## STEP 1A.1 — Basic outlier screening (annual_income)

In [0]:

df.select(
    F.min(F.col("financials.annual_income").cast("double")).alias("min_annual_income"),
    F.max(F.col("financials.annual_income").cast("double")).alias("max_annual_income")
).display()

min_annual_income,max_annual_income
0.0,171000.0


 Extreme values were screened using min and max statistics to identify potential outliers.



##STEP 1B — UNIQUENESS CHECK


Duplicate primary keys were detected in the dataset.
Two application IDs, app_042 and app_001, each appeared twice.

Since _id represents the unique identifier of a credit application, duplicates violate the uniqueness dimension of data quality. This can lead to:

- Inflated approval rates
- Biased model training
- Incorrect aggregate statistics

To preserve entity integrity, duplicate records were removed using dropDuplicates(["_id"]).

The deduplicated dataset is used for all subsequent quality and bias analysis steps.

In [0]:

# Count duplicates
duplicate_ids = (
    df.groupBy("_id")
      .count()
      .filter(F.col("count") > 1)
)

# Show duplicate IDs if any
display(duplicate_ids)

# Count how many duplicate IDs exist
print("Number of duplicate IDs:", duplicate_ids.count())

_id,count
app_042,2
app_001,2


Number of duplicate IDs: 2


## STEP 1B.1 — Remediation: remove duplicates by primary key (_id)

In [0]:

rows_before = df.count()
df_dedup = df.dropDuplicates(["_id"])
rows_after = df_dedup.count()

print("Rows before deduplication:", rows_before)
print("Rows after deduplication:", rows_after)

# Use deduplicated df for the rest of the checks
df = df_dedup

Rows before deduplication: 502
Rows after deduplication: 500


Duplicates were removed using dropDuplicates on _id. The deduplicated dataset is used for subsequent checks




## STEP 1C — CONSISTENCY CHECK



This step evaluates categorical consistency in the dataset.

The gender attribute is standardized to ensure:
- Lowercase formatting
- Removal of leading and trailing spaces
- No inconsistent category representations

Consistency ensures reliable aggregation and fair bias analysis.



## STEP 1C — CONSISTENCY CHECK (Gender standardization)


In [0]:
from pyspark.sql import functions as F



df_consistency = df.withColumn(
    "gender_clean",
    F.lower(F.trim(F.col("applicant_info.gender")))
)

df_consistency.groupBy("gender_clean").count().orderBy(F.desc("count")).display()

gender_clean,count
male,194
female,193
f,58
m,53
,2


## Result — Gender Consistency

The gender distribution after lowercase and trimming shows multiple representations of the same category:

- male → 195
- m → 53
- female → 193
- f → 58
- empty string → 2
- null → 1

This indicates inconsistent categorical encoding, since `male` and `m` represent the same category, and `female` and `f` represent the same category.

Such inconsistencies can:

- Distort group-based statistics
- Bias fairness analysis
- Produce misleading aggregation results

Therefore, the gender attribute requires normalization.

============================================================

STEP 1D — CONSISTENCY CHECK

============================================================

## STEP 1D - CONSISTENCY CHECK (Gender) 

Purpose: inspect category encoding for gender (case, whitespace, short forms)

In [0]:

from pyspark.sql import functions as F

# Select nested gender field and give it a simple column name
gender_df = df.select(F.col("applicant_info.gender").alias("gender_raw"))

# Show raw values distribution (before cleaning)
gender_df.groupBy("gender_raw").count().orderBy(F.desc("count")).display()

gender_raw,count
Male,194
Female,193
F,58
M,53
,2


## STEP 1D.1 — Cross-field consistency check

 Approved loans should have an approved_amount value

In [0]:

inconsistent_approval = df.filter(
    (F.col("decision.loan_approved") == True) &
    (F.col("decision.approved_amount").isNull())
)

print("Inconsistent approved records (loan_approved=True but approved_amount is null):",
      inconsistent_approval.count())

display(inconsistent_approval.select("_id", "decision.loan_approved", "decision.approved_amount").limit(10))

Inconsistent approved records (loan_approved=True but approved_amount is null): 0


_id,loan_approved,approved_amount


	•	Eğer count 0 ise: “No cross-field inconsistencies were detected.”
	•	Eğer >0 ise: “These records were flagged for remediation or exclusion

In [0]:
# ------------------------------------------------------------
# Normalize gender values to check consistency after cleaning
# ------------------------------------------------------------

gender_clean_df = df.select(
    F.lower(F.trim(F.col("applicant_info.gender"))).alias("gender_clean")
)

gender_clean_df.groupBy("gender_clean").count().orderBy(F.desc("count")).display()

gender_clean,count
male,194
female,193
f,58
m,53
,2


## Result — Post-Standardization Gender Check

After applying lowercase transformation and trimming, formatting inconsistencies were removed.

However, semantic inconsistencies remain:

- male → 195
- m → 53
- female → 193
- f → 58
- empty string → 2
- null → 1

Short forms ("m", "f") still represent the same categories as
"male" and "female".

Therefore, an additional normalization step is required to map:

- "m" → "male"
- "f" → "female"

Empty and null values should be handled separately.


##STEP 1E — ACCURACY CHECK (FORMAT VALIDATION)



In [0]:
# ------------------------------------------------------------
# 1E - Accuracy
# Validate format of email and SSN
# ------------------------------------------------------------

accuracy_check = df.select(
    F.sum(
        (~F.col("applicant_info.email")
           .rlike("^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"))
        .cast("int")
    ).alias("invalid_email_format"),

    F.sum(
        (~F.col("applicant_info.ssn")
           .rlike("^[0-9]{3}-[0-9]{2}-[0-9]{4}$"))
        .cast("int")
    ).alias("invalid_ssn_format")
)

accuracy_check.display()

invalid_email_format,invalid_ssn_format
11,0


## Result — Accuracy (Format Validation)

The format validation shows:

- Invalid email formats: 11  
- Invalid SSN formats: 0  

This indicates that all SSN values follow the expected pattern (XXX-XX-XXXX),  
while 11 email records do not match the standard email structure.

Invalid email formats may:
- Reduce communication reliability
- Affect downstream identity validation processes
- Introduce data integrity risks

These records should be flagged or corrected during remediation.


##STEP 2 — REMEDIATION (CLEAN VERSION)



In [0]:
# ------------------------------------------------------------
# Standardize Gender
# ------------------------------------------------------------

df_clean = df.withColumn(
    "gender_standardized",
    F.when(F.lower(F.col("applicant_info.gender")).isin("m", "male"), "Male")
     .when(F.lower(F.col("applicant_info.gender")).isin("f", "female"), "Female")
     .otherwise("Unknown")
)

# ------------------------------------------------------------
# Cast annual_income to double
# ------------------------------------------------------------

df_clean = df_clean.withColumn(
    "annual_income_double",
    F.col("financials.annual_income").cast("double")
)

display(df_clean.limit(5))

_id,applicant_info,decision,financials,loan_purpose,notes,processing_timestamp,spending_behavior,gender_standardized,annual_income_double
app_307,"List(1990/07/26, joseph.lee69@aol.com, Joseph Lee, M, 10.189.246.72, 178-35-6543, 10097)","List(57000, 6.1, true, null)","List(53000, null, 83, 0.1, 17420)",null,null,null,"List(List(482, Shopping))",Male,53000.0
app_133,"List(1989-10-07, frank.harris11@outlook.com, Frank Harris, Male, 172.25.65.254, 507-24-3732, 10022)","List(null, null, false, algorithm_risk_score)","List(70000, null, 72, 0.16, 33883)",null,null,null,"List(List(319, Insurance), List(407, Dining))",Male,70000.0
app_130,"List(03/10/1981, jennifer.wright10@mail.com, Jennifer Wright, Female, 172.27.3.208, 286-43-9771, 90252)","List(null, null, false, algorithm_risk_score)","List(135000, null, 62, 0.09, 30902)",null,null,null,"List(List(804, Healthcare), List(671, Fitness), List(598, Rent))",Female,135000.0
app_404,"List(1985-11-10, thomas.mitchell21@aol.com, Thomas Mitchell, Male, 172.30.59.137, 543-21-6993, 10017)","List(69000, 5.9, true, null)","List(91000, null, 31, 0.32, 2729)",null,null,null,"List(List(841, Insurance), List(325, Fitness))",Male,91000.0
app_407,"List(1990/11/09, jacob.ramirez30@aol.com, Jacob Ramirez, Male, 192.168.53.145, 458-36-1005, 10076)","List(null, null, false, algorithm_risk_score)","List(74000, null, 31, 0.26, 32341)",null,null,null,"List(List(52, Entertainment), List(606, Fitness), List(781, Insurance))",Male,74000.0


In [0]:
df_clean.select(
    "_id",
    "gender_standardized",
    "annual_income_double"
).display()

_id,gender_standardized,annual_income_double
app_001,Female,102000.0
app_002,Male,41000.0
app_003,Female,65000.0
app_004,Female,69000.0
app_005,Female,39000.0
app_006,Male,82000.0
app_007,Female,92000.0
app_008,Female,80000.0
app_009,Female,92000.0
app_010,Female,44000.0


## STEP 2 — Final clean dataset preview (deliverable)

In [0]:

# If you created df_clean earlier, keep it.
# Otherwise, define it now as the latest cleaned dataframe.
df_clean = df

df_clean.printSchema()
display(df_clean.limit(5))

print("Final clean row count:", df_clean.count())

root
 |-- _id: string (nullable = true)
 |-- applicant_info: struct (nullable = true)
 |    |-- date_of_birth: string (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- full_name: string (nullable = true)
 |    |-- gender: string (nullable = true)
 |    |-- ip_address: string (nullable = true)
 |    |-- ssn: string (nullable = true)
 |    |-- zip_code: string (nullable = true)
 |-- decision: struct (nullable = true)
 |    |-- approved_amount: long (nullable = true)
 |    |-- interest_rate: double (nullable = true)
 |    |-- loan_approved: boolean (nullable = true)
 |    |-- rejection_reason: string (nullable = true)
 |-- financials: struct (nullable = true)
 |    |-- annual_income: string (nullable = true)
 |    |-- annual_salary: long (nullable = true)
 |    |-- credit_history_months: long (nullable = true)
 |    |-- debt_to_income: double (nullable = true)
 |    |-- savings_balance: long (nullable = true)
 |-- loan_purpose: string (nullable = true)
 |-- notes: stri

_id,applicant_info,decision,financials,loan_purpose,notes,processing_timestamp,spending_behavior
app_307,"List(1990/07/26, joseph.lee69@aol.com, Joseph Lee, M, 10.189.246.72, 178-35-6543, 10097)","List(57000, 6.1, true, null)","List(53000, null, 83, 0.1, 17420)",null,null,null,"List(List(482, Shopping))"
app_133,"List(1989-10-07, frank.harris11@outlook.com, Frank Harris, Male, 172.25.65.254, 507-24-3732, 10022)","List(null, null, false, algorithm_risk_score)","List(70000, null, 72, 0.16, 33883)",null,null,null,"List(List(319, Insurance), List(407, Dining))"
app_130,"List(03/10/1981, jennifer.wright10@mail.com, Jennifer Wright, Female, 172.27.3.208, 286-43-9771, 90252)","List(null, null, false, algorithm_risk_score)","List(135000, null, 62, 0.09, 30902)",null,null,null,"List(List(804, Healthcare), List(671, Fitness), List(598, Rent))"
app_404,"List(1985-11-10, thomas.mitchell21@aol.com, Thomas Mitchell, Male, 172.30.59.137, 543-21-6993, 10017)","List(69000, 5.9, true, null)","List(91000, null, 31, 0.32, 2729)",null,null,null,"List(List(841, Insurance), List(325, Fitness))"
app_407,"List(1990/11/09, jacob.ramirez30@aol.com, Jacob Ramirez, Male, 192.168.53.145, 458-36-1005, 10076)","List(null, null, false, algorithm_risk_score)","List(74000, null, 31, 0.26, 32341)",null,null,null,"List(List(52, Entertainment), List(606, Fitness), List(781, Insurance))"


Final clean row count: 500


## Final Data Quality Assessment Conclusion

The final cleaned dataset (df_clean) represents the output of the data quality remediation process and serves as the foundation for downstream bias analysis.

Following remediation, the dataset satisfies the core data quality dimensions of completeness, uniqueness, validity, consistency, and accuracy for analytical and bias evaluation purposes.

Duplicate primary keys were removed to preserve entity integrity. Gender values were standardized to eliminate categorical inconsistencies. Financial attributes were cast to appropriate data types to ensure numerical validity. Email and SSN fields were validated using format constraints. Logical range checks confirmed that critical financial indicators fall within acceptable boundaries.

---

## Additional Structural Observations

While the dataset is analytically suitable, further inspection revealed structural characteristics that introduce technical risk in production environments.

### 1. Inconsistent Date of Birth Format

The `date_of_birth` field appears in multiple formats:

- YYYY-MM-DD  
- YYYY/MM/DD  
- MM/DD/YYYY  

Although these values are not missing, inconsistent temporal formatting increases technical risk. It may lead to incorrect age calculation, parsing inconsistencies, feature engineering errors, and instability in automated pipelines.

Although this does not invalidate analytical results, inconsistent date formatting should be standardized in any scalable data architecture.

---

### 2. Nested JSON Structure

The dataset retains nested schemas within:

- applicant_info  
- financials  
- decision  
- spending_behavior  

While structurally valid, nested schemas increase transformation complexity, complicate aggregation logic, and introduce schema evolution risks in production pipelines.

Flattening or schema normalization would be recommended for large-scale modeling or deployment scenarios.

---

### 3. High Null Density in Non-Critical Fields

Fields such as:

- loan_purpose  
- notes  
- processing_timestamp  

Contain a high proportion of null values.

These attributes are not critical for bias analysis and therefore do not compromise the analytical objectives of this assessment. However, they may limit explanatory depth in advanced modeling workflows.

---

## Risk Assessment

The dataset is analytically robust and suitable for bias evaluation following the remediation steps performed.

However, for production-grade deployment, automated decision systems, or scalable data engineering pipelines, additional temporal standardization and schema normalization would be required to reduce technical risk and ensure long-term maintainability.

##  Data Quality Summary

The dataset was evaluated across five core data quality dimensions:

- **Completeness:** Minor missing values were detected in selected financial fields, with no missing values in the target variable (loan_approved).
- **Uniqueness:** Two duplicate application IDs were identified and removed to preserve entity integrity.
- **Validity:** Logical range checks identified a small number of inconsistencies in debt-to-income ratios and credit history values.
- **Consistency:** Gender categories contained multiple representations (e.g., "Male", "M", "Female", "F") and were standardized.
- **Accuracy:** 11 records contained invalid email formats, while SSN formats were valid across all records.

After remediation, duplicate records were removed, categorical fields were standardized, and type casting was applied where necessary.  
The cleaned dataset is considered analytically reliable and suitable for downstream bias and fairness analysis.

## Conclusion

This assessment systematically evaluated the NovaCred credit applications dataset across five core data quality dimensions: completeness, uniqueness, validity, consistency, and accuracy.

The analysis identified structural and categorical inconsistencies, including duplicate primary keys, non-standardized gender values, missing financial attributes, and formatting irregularities in selected fields. While most logical validity checks passed predefined business constraints, categorical normalization and format validation were required to ensure analytical robustness.

Following controlled remediation steps, including deduplication, categorical standardization, and structural validation, the dataset now satisfies core governance requirements and is suitable for downstream bias analysis and statistical evaluation.